# Statistical Fallacies: Simpson, Survivorship, p-Hacking, Base Rates and Regression to the Mean

**Why this matters.** Most bad data-science conclusions do not come from a coding bug but from a reasoning trap: comparing groups that are not comparable, looking only at survivors, testing until something works, forgetting how rare a
condition is, or crediting an intervention for a change that would have happened anyway. You will meet each of these at work, and interviewers love them. Here every fallacy is **reproduced by simulation or data**, so you can recognise its signature.

**You will learn**
- detect **Simpson's paradox** (a trend that reverses when you split by a confounder) and decide which view to trust
- reproduce **survivorship bias**, **p-hacking** and **regression to the mean** with simulations
- avoid **base-rate neglect** when judging alerts and classifiers
- build a short checklist of questions to ask about any surprising result

**Prerequisites:** [Hypothesis testing](06_hypothesis_testing.ipynb), [A/B testing](07_ab_testing_end_to_end.ipynb) and [Probability and simulation](02_probability_and_simulation.ipynb).

> Data used: `churn.csv` (synthetic, see `docs/DATASETS.md`) and constructed or simulated examples where the fallacy needs a known ground truth.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

DATA = Path("../datasets")
rng = np.random.default_rng(42)
churn = pd.read_csv(DATA / "churn.csv")
churn["contract"] = churn["contract"].str.lower()

## 1. Simpson's paradox

Two treatments for a condition. Treatment A is given mostly to severe cases, B mostly to mild ones. (Constructed numbers, chosen so the reversal is exact.)

In [ ]:
tbl = pd.DataFrame({
    "treatment": ["A", "A", "B", "B"],
    "severity": ["mild", "severe", "mild", "severe"],
    "patients": [100, 1000, 1000, 100],
    "recovered": [90, 600, 800, 50],
})
tbl["rate"] = tbl["recovered"] / tbl["patients"]
print(tbl.to_string(index=False))

overall = tbl.groupby("treatment")[["recovered", "patients"]].sum()
overall["rate"] = overall["recovered"] / overall["patients"]
print("\noverall recovery rate:\n", overall["rate"].round(3).to_string())

*What just happened?* A has the **higher** recovery rate in *both* severity groups (90% vs 80% in mild cases, 60% vs 50% in severe ones), yet the **lower** rate overall. The overall rate is a weighted average, and the weights differ:
most of A's patients are severe (low recovery for any treatment) while most of B's are mild. Severity is a **confounder**: it influences both which treatment a patient got and the outcome.

**Which view is right?** It depends on the causal story, not on the numbers. If severity affects the choice of treatment (doctors give A to sicker patients), the stratified comparison is the fair one. If instead the stratifying variable were a *consequence*
of the treatment (say, "recovered within a week"), conditioning on it would be wrong. Random assignment removes the problem: with randomisation, severity is balanced across arms.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
w = 0.35
cats = ["mild", "severe", "overall"]
ra = [tbl.query("treatment=='A' and severity==@c")["rate"].iloc[0] for c in ["mild", "severe"]] + [overall.loc["A", "rate"]]
rb = [tbl.query("treatment=='B' and severity==@c")["rate"].iloc[0] for c in ["mild", "severe"]] + [overall.loc["B", "rate"]]
ax.bar(np.arange(3) - w / 2, ra, w, label="A"); ax.bar(np.arange(3) + w / 2, rb, w, label="B")
ax.set_xticks(range(3), cats); ax.set_ylabel("recovery rate"); ax.set_title("A wins in every subgroup and loses overall"); ax.legend()
plt.show()

**Checking your own data.** Whenever you compare groups, ask whether the groups differ in composition. With the churn data: does the apparent relation between contract and churn hold within internet-service groups?
(Here it does; not every comparison reverses. The habit is to *look*.)

In [ ]:
c2 = churn.assign(internet_service=churn["internet_service"].fillna("No internet"))
print(c2.pivot_table(index="contract", columns="internet_service", values="churned", aggfunc="mean").round(3))

Within each internet type the ordering month-to-month > one year > two year stays the same, so the pattern is not a composition artifact.

## 2. Survivorship bias

Looking only at what "survived" a filter and forgetting the ones that dropped out. Classic: average returns of funds that still exist. Simulate 2,000 funds whose true mean yearly return is exactly 5% (pure luck around it); funds whose cumulative return falls below -30% shut down.

In [ ]:
years, n_funds = 10, 2000
ret = rng.normal(0.05, 0.20, size=(n_funds, years))
cum = np.cumprod(1 + ret, axis=1)
dead = (cum < 0.7).any(axis=1)                                 # fell below 70% of initial capital at some point -> closed
print(f"true mean yearly return of every fund: 5.0% | funds closed: {dead.mean():.0%}")
print(f"average yearly return over ALL funds        : {ret.mean():.2%}")
print(f"average yearly return over SURVIVING funds  : {ret[~dead].mean():.2%}   <- looks like skill, is only filtering")

Nobody has skill here, yet the survivors average a higher return because the losers were removed from the list. The same trap appears when you study "our current customers" to learn why people stay, analyse only users who completed onboarding,
or read success stories of companies that did X. In the churn table, current customers have longer tenure by construction; fairly comparing retention requires tracking *everyone who started*, including those who left.

**Antidote:** ask "what is missing from this dataset and why?" and reconstruct the full starting population.

## 3. p-hacking and the garden of forking paths

If you test many hypotheses and report the ones with `p < 0.05`, you will "discover" effects in pure noise. Below, a random outcome and 100 random, unrelated predictors on 200 rows:

In [ ]:
n, k = 200, 100
y = rng.normal(size=n)
X = rng.normal(size=(n, k))
pvals = np.array([stats.pearsonr(X[:, j], y)[1] for j in range(k)])
print(f"'significant' at 0.05 among {k} pure-noise predictors: {(pvals < 0.05).sum()}  (expected about {0.05 * k:.0f})")
best = pvals.argmin()
print(f"best predictor #{best}: r = {stats.pearsonr(X[:, best], y)[0]:+.3f}, p = {pvals[best]:.4f}")
print(f"after Bonferroni (p < 0.05/{k}): {(pvals < 0.05 / k).sum()} significant")

The "best" predictor looks like a finding. Now check it on fresh data from the same process. A real effect would persist; a fluke will not:

In [ ]:
y2 = rng.normal(size=n); X2 = rng.normal(size=(n, k))
print(f"same predictor on new data: r = {stats.pearsonr(X2[:, best], y2)[0]:+.3f}, p = {stats.pearsonr(X2[:, best], y2)[1]:.3f}")

How often does a researcher who tries 40 predictors (on 100 rows) find at least one `p < 0.05`? Repeat the exercise 2,000 times:

In [ ]:
k = 40
found = 0
for _ in range(2000):
    yy = rng.normal(size=100); XX = rng.normal(size=(100, k))
    r = (XX - XX.mean(0)).T @ (yy - yy.mean()) / (100 * XX.std(0) * yy.std())
    t = r * np.sqrt(98 / (1 - r**2)); p = 2 * stats.t.sf(np.abs(t), 98)
    found += (p < 0.05).any()
print(f"share of pure-noise studies with at least one p < 0.05: {found / 2000:.2f}   (theory: 1 - 0.95^{k} = {1 - 0.95 ** k:.2f})")

Forking paths also hide in *analysis choices*: which outliers to drop, which subgroup, which transform, which test. Defences: pre-register the primary analysis, correct for multiple comparisons, hold out data for confirmation, and report *everything* you tried.

## 4. Base-rate neglect

Recap of the core idea from the probability notebook with a new setting. A churn-risk model flags customers with 90% sensitivity and 90% specificity. Suppose only 2% of the customers you score are about to churn
(for example, a list of customers already on a long contract). What share of flagged customers will actually churn?

In [ ]:
prev, sens, spec = 0.02, 0.90, 0.90
precision = sens * prev / (sens * prev + (1 - spec) * (1 - prev))
print(f"precision (share of flagged who really churn) = {precision:.1%}")
N = 500_000
truth = rng.random(N) < prev
flag = np.where(truth, rng.random(N) < sens, rng.random(N) < (1 - spec))
print(f"simulation check: {truth[flag].mean():.1%} of {flag.sum():,} flagged customers truly churn")

A model that sounds excellent (90%/90%) produces alerts that are wrong most of the time when the event is rare. Always ask for **precision at the real-world base rate**, not only sensitivity/accuracy. (On the churn dataset the base rate is 37%, where the same model would be far more useful.)

## 5. Regression to the mean

Any measurement is skill/true level plus luck. Pick the extreme performers on the first measurement and, on the second, they will on average look *less extreme*, because part of what made them extreme was luck that does not repeat.
No intervention is needed. Simulate exam scores = true ability + noise, then "tutor" the bottom 10% after test 1, with a tutoring effect of exactly zero:

In [ ]:
n = 5000
ability = rng.normal(70, 8, n)
test1 = ability + rng.normal(0, 8, n)
test2 = ability + rng.normal(0, 8, n)
bottom = test1 <= np.quantile(test1, 0.10)
print(f"bottom 10% on test 1: mean {test1[bottom].mean():.1f} -> on test 2: mean {test2[bottom].mean():.1f}  (tutoring had NO effect)")
print(f"correlation between the two tests = {np.corrcoef(test1, test2)[0, 1]:.2f}")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(test1[::5], test2[::5], s=5, alpha=0.4)
ax.scatter(test1[bottom], test2[bottom], s=6, color="tab:red", label="bottom 10% on test 1")
lims = [30, 110]; ax.plot(lims, lims, "k--", label="no change")
ax.set_xlabel("test 1 score"); ax.set_ylabel("test 2 score"); ax.set_title("Extremes drift toward the average on retest"); ax.legend()
plt.show()

The improvement equals roughly `(1 - r) x` the initial gap from the mean, where `r` is the test-retest correlation. A company that gives a retention offer to "customers whose usage dropped this week" and sees usage recover will credit the offer, but much
of the recovery is regression to the mean. The fix is a **control group** chosen by the same rule.

## A checklist for surprising results

1. Are the groups comparable? What confounders differ? (Simpson)
2. What is missing from the dataset, and why? (survivorship, selection)
3. How many things were tried before this one was reported? (p-hacking)
4. How rare is the thing being predicted? (base rate)
5. Was the group selected because of an extreme value? Is there a control? (regression to the mean)
6. Does the effect persist on new data?

## Common mistakes

1. **Reporting only the aggregate** without checking composition by the obvious confounders.
2. **Adjusting for everything**, including variables affected by the treatment, which can create bias rather than remove it.
3. **Learning from successes only.**
4. **Running many analyses and reporting the best**; or peeking and stopping at the first significant result.
5. **Judging an alert system by sensitivity alone.**
6. **Declaring a fix effective after it was applied to the worst cases only**, with no control.

In [ ]:
# Mistake 2: control the wrong variable. True story: X -> M -> Y. Conditioning on the mediator M hides the real effect of X
x = rng.normal(size=5000); m = x + rng.normal(0, 0.3, 5000); yv = m + rng.normal(0, 1, 5000)
import statsmodels.api as sm
print("effect of X on Y (total):            ", round(sm.OLS(yv, sm.add_constant(x)).fit().params[1], 2))
print("effect of X on Y 'controlling' for M:", round(sm.OLS(yv, sm.add_constant(np.column_stack([x, m]))).fit().params[1], 2), "<- wrongly suggests X does nothing")

## Interview corner

**1. Explain Simpson's paradox and how you would decide which analysis to trust.**
<details><summary>Answer</summary>
A trend in aggregated data can reverse within subgroups when a confounder is unevenly distributed across the compared groups. Decide using the causal structure: adjust for confounders that affect both exposure and outcome, do not condition on consequences of the exposure. Randomised experiments sidestep the issue.
</details>

**2. A fund manager shows that the average of their funds beat the market for 10 years. What would you ask?**
<details><summary>Answer</summary>
Are closed or merged funds included (survivorship)? How were these funds chosen, and after the fact? How many other funds/strategies were tried (multiple comparisons)? Is performance risk-adjusted, and does it persist out of sample?
</details>

**3. What is p-hacking and how do you guard against it?**
<details><summary>Answer</summary>
Trying many analyses, subgroups or metrics and reporting those with p &lt; 0.05. Guard with a pre-registered primary metric and analysis plan, multiple-comparison corrections, hold-out confirmation, and transparent reporting of everything tried.
</details>

**4. Why can a retention campaign targeted at the lowest-engagement users look effective even if it does nothing?**
<details><summary>Answer</summary>
Regression to the mean: users selected for unusually low engagement will on average look better next period from chance alone. Compare with a control group selected by the same rule and randomly left untreated.
</details>

## Exercises

### Exercise 1: Find the reversal
Below is a constructed table of click-through on two ad designs across two audiences. Compute the **overall** click rate per design (`overall_rate`, a Series indexed by design) and the rate **per audience** (`by_audience`, a DataFrame with audiences as index
and designs as columns). Then store `reversal` = True if one design wins overall while the other wins in **both** audiences.

In [ ]:
ads = pd.DataFrame({
    "design": ["X", "X", "Y", "Y"],
    "audience": ["new", "returning", "new", "returning"],
    "impressions": [2000, 500, 500, 2000],
    "clicks": [100, 125, 20, 400],
})

In [ ]:
overall_rate = ...
by_audience = ...
reversal = ...

In [ ]:
o = ads.groupby("design")[["clicks", "impressions"]].sum()
exp_o = o["clicks"] / o["impressions"]
assert np.allclose(overall_rate.sort_index(), exp_o.sort_index()), "overall rate = total clicks / total impressions per design (not the mean of the rates)"
ba = ads.assign(rate=ads["clicks"] / ads["impressions"]).pivot(index="audience", columns="design", values="rate")
assert np.allclose(by_audience.loc[ba.index, ba.columns].to_numpy(), ba.to_numpy()), "by_audience: rows = audience, columns = design, values = click rate"
exp_rev = bool((exp_o["X"] > exp_o["Y"]) and (ba["X"] < ba["Y"]).all() or (exp_o["X"] < exp_o["Y"]) and (ba["X"] > ba["Y"]).all())
assert bool(reversal) == exp_rev
print(f"Exercise 1 passed: overall {exp_o.round(3).to_dict()}, reversal = {reversal}")

In [ ]:
_o = ads.groupby("design")[["clicks", "impressions"]].sum()
overall_rate = _o["clicks"] / _o["impressions"]
by_audience = ads.assign(rate=ads["clicks"] / ads["impressions"]).pivot(index="audience", columns="design", values="rate")
reversal = bool(((overall_rate["X"] > overall_rate["Y"]) and (by_audience["X"] < by_audience["Y"]).all())
                or ((overall_rate["X"] < overall_rate["Y"]) and (by_audience["X"] > by_audience["Y"]).all()))

### Exercise 2: How many false discoveries?
Generate 60 pure-noise predictors: `rs = np.random.default_rng(123)`, `y_n = rs.normal(size=150)`, `X_n = rs.normal(size=(150, 60))`. For each column compute the Pearson p-value against `y_n`. Store `n_sig` (p < 0.05), `n_bonf` (p < 0.05/60)
and `min_p` (the smallest p-value).

In [ ]:
rs = np.random.default_rng(123)
y_n = rs.normal(size=150)
X_n = rs.normal(size=(150, 60))

In [ ]:
n_sig = ...
n_bonf = ...
min_p = ...

In [ ]:
pv = np.array([stats.pearsonr(X_n[:, j], y_n)[1] for j in range(60)])
assert n_sig == int((pv < 0.05).sum()), "count p < 0.05"
assert n_bonf == int((pv < 0.05 / 60).sum()), "count p < 0.05/60"
assert abs(min_p - pv.min()) < 1e-12
print(f"Exercise 2 passed: {n_sig} 'significant' of 60 noise columns, {n_bonf} after Bonferroni, smallest p = {min_p:.4f}")

In [ ]:
_pv = np.array([stats.pearsonr(X_n[:, j], y_n)[1] for j in range(60)])
n_sig = int((_pv < 0.05).sum())
n_bonf = int((_pv < 0.05 / 60).sum())
min_p = float(_pv.min())

### Exercise 3: Predict the regression to the mean
Using the exam simulation (`test1`, `test2`, `bottom`), store in `actual_gain` the mean of `test2 - test1` for the bottom 10%. Then store in `predicted_gain` the prediction `(1 - r) * (mean(test1) - mean(test1[bottom]))`, where `r` is the correlation of
`test1` and `test2`. They should agree within 1.0 point.

In [ ]:
actual_gain = ...
predicted_gain = ...

In [ ]:
r_ = np.corrcoef(test1, test2)[0, 1]
assert abs(actual_gain - (test2[bottom] - test1[bottom]).mean()) < 1e-9, "mean of (test2 - test1) over the bottom-10% group"
assert abs(predicted_gain - (1 - r_) * (test1.mean() - test1[bottom].mean())) < 1e-9, "(1 - r) * gap from the overall mean"
assert actual_gain > 0 and abs(actual_gain - predicted_gain) < 1.0, "regression to the mean predicts the gain within about a point"
print(f"Exercise 3 passed: actual gain {actual_gain:.2f}, predicted {predicted_gain:.2f}")

In [ ]:
actual_gain = float((test2[bottom] - test1[bottom]).mean())
_r = np.corrcoef(test1, test2)[0, 1]
predicted_gain = float((1 - _r) * (test1.mean() - test1[bottom].mean()))

### Exercise 4 (open-ended): Hunt for a fallacy in the wild
Pick a claim you have seen at work or in the news ("customers who use feature X churn less", "our top-decile agents are getting worse", "the model has 95% accuracy"). Write the claim, name which fallacy from this notebook might apply,
and describe the data or experiment that would settle it. If you have a dataset, simulate the fallacy on it.

## Recap
- Simpson's paradox: aggregates hide composition; the right view follows from the causal story, and randomisation avoids it.
- Survivorship bias: the data you have is a filtered sample; ask what is missing.
- p-hacking: many tests guarantee false discoveries; pre-register, correct, and confirm on held-out data.
- Base-rate neglect: precision collapses when the event is rare, however good sensitivity and specificity look.
- Regression to the mean: extremes on one measurement drift back on the next; use controls.

**Go deeper:** "Thinking, Fast and Slow" (Kahneman) for base rates and regression; Judea Pearl's "The Book of Why" for Simpson's paradox and causal reasoning; Gelman and Loken's paper "The garden of forking paths".

**Next:** you have finished the statistics module. Continue with the machine learning module; see the [module README](README.md) for what to try next.